# Part 3: ensemble of recurrent location extractors

Flow: **members → probability bundles → recipe search on test-like data → submission → actual LB score**.

* Members are trained elsewhere (`A3_colab.ipynb` / `part2.train`, logged to W&B). Here a member is only a checkpoint plus a loader.
* Every member is run once per split and its O/B/I probabilities are cached as a bundle on Drive. Combinations never retrain anything.
* The recipe (members, weights, threshold) is chosen on `testlike_select` only. `testlike_report`, the 26 organiser examples, checkpoint and calibration are reported for every candidate. Audit is scored once, after freezing.

In [ ]:
from pathlib import Path
import os, sys

COLAB = Path('/content').exists()
if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/KSE_AI301_A3') if COLAB else Path('.').resolve()
PROJECT = DRIVE/'project' if COLAB else DRIVE          # repo checkout (ensemble branch)
DATA = DRIVE/'data'
RESULTS = DRIVE/'results'/'ensemble'
BUNDLES = RESULTS/'bundles'
SPLIT = PROJECT/'part2_split.json'
ENSEMBLE_SPLIT = PROJECT/'ensemble_split.json'
sys.path.insert(0, str(PROJECT)); os.chdir(PROJECT)
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT/'requirements-part2.txt')])
# os.environ['WANDB_PROJECT'] = '...'   # same project as the members; needs WANDB_API_KEY

In [ ]:
# One entry per candidate member. `expected` reproduces a score logged for that checkpoint;
# export stops if the bundle disagrees (wrong checkpoint, loader or tokenizer).
MEMBERS = [
    dict(name='word_bilstm', kind='part2', checkpoint=DRIVE/'part2_model'/'checkpoint.pt',
         expected=dict(split='calibration', threshold=.8, policy='strict', f1=0.7952)),
    # dict(name='bpe_bilstm', kind='part2', checkpoint=DRIVE/'members'/'bpe-main'/'best.pt',
    #      expected=dict(split='calibration', threshold=.8, policy='strict', f1=0.82196078)),
    # dict(name='ngrams_bilstm', kind='ngrams', checkpoint=DRIVE/'members'/'ngrams-main'/'best.pt'),  # needs n-gram loader
]
# Checkpoints logged as W&B artifacts can be pulled first:
# from ensemble_members import fetch_wandb_artifact
# fetch_wandb_artifact('entity/project/artifact-name:v0', DRIVE/'members'/'ngrams-main')

## 1. Frozen test-like split (build once, then commit `ensemble_split.json`)

The rows most test-like by an adversarial classifier, taken from labelled rows outside every Part 2 partition. Members trained on more data must exclude these IDs.

In [ ]:
from ensemble_data import build_ensemble_split, read_ensemble_split
if not ENSEMBLE_SPLIT.exists():
    build_ensemble_split(DATA, SPLIT, ENSEMBLE_SPLIT, pool_rows=200000, testlike_rows=2000)
spec = read_ensemble_split(ENSEMBLE_SPLIT, SPLIT)
{k: spec[k] for k in ['pool_rows', 'adversarial_auc', 'score_min', 'score_median']}

In [ ]:
from ensemble_data import load_docs
SPLITS = ['testlike_select', 'testlike_report', 'labeling', 'checkpoint', 'calibration', 'test']
docs = load_docs(DATA, SPLITS, SPLIT, ENSEMBLE_SPLIT)
{k: len(v) for k, v in docs.items()}

## 2. Probability bundles (one pass per member and split; resumable)

In [ ]:
import torch
from ensemble_members import export_member
device = 'cuda' if torch.cuda.is_available() else 'cpu'
for member in MEMBERS:
    print(member['name'], 'verified F1:', export_member(member, docs, BUNDLES, device))

## 3. Recipe search

In [ ]:
from ensemble import run_search, log_wandb
SEARCH = RESULTS/'search_v1'          # new folder for every change of members or settings
names = [m['name'] for m in MEMBERS]
result, report = run_search(BUNDLES, names, docs, SEARCH, rounds=10, max_size=4)
log_wandb(SEARCH.name, SEARCH)
result['final']

In [ ]:
report.pivot_table(index='recipe', columns='split', values='f1').sort_values('testlike_select', ascending=False)

## 4. Submission and the actual public score

In [ ]:
from ensemble import export_test
final = result['final']
submission = export_test(final, BUNDLES, docs['test'], SEARCH/'submission.csv')
submission

In [ ]:
# After uploading to Kaggle, record the real public score (never a guess):
# from ensemble import record_submission
# record_submission(RESULTS/'leaderboard.csv', final, submission, public_f1=0.0, note='')

## 5. Audit (once, after the recipe is frozen)

In [ ]:
# from ensemble import score_audit
# audit = load_docs(DATA, ['audit'], SPLIT, allow_audit=True)['audit']
# for member in MEMBERS: export_member(member, dict(audit=audit), BUNDLES, device)
# score_audit(final, BUNDLES, audit, SEARCH)